In [2]:
import os
import tiktoken  # 替换 langchain 的分块器
from typing import List, Dict, Any
from chromadb import PersistentClient, Collection
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
import re
import requests
import json

In [3]:

# 配置参数
CHROMA_DB_PATH = "./chroma_local_db"
EMBEDDING_MODEL = "/home/dell/Downloads/bge-m3/"
CHUNK_SIZE = 500  # 文本块字符数
CHUNK_OVERLAP = 0  # 块重叠字符数
TOP_K = 5

class ChromaKBManager:
    def __init__(self):
        self.client = PersistentClient(path=CHROMA_DB_PATH)
        self.embedding_function = SentenceTransformerEmbeddingFunction(
            model_name=EMBEDDING_MODEL,
            device="cuda"
        )
        # 初始化 tiktoken 分块器（支持中英文）
        self.encoder = tiktoken.get_encoding("cl100k_base")  # 通用编码模型

    def list_all_collections(self) -> List[str]:
        """获取所有集合的名称列表"""
        collections = self.client.list_collections()
        return [col.name for col in collections]

    def get_collection_document_count(self, collection_name: str) -> int:
        """获取指定集合中的文档数量"""
        try:
            collection = self.client.get_collection(name=collection_name)
            return collection.count()  # 返回文档总数
        except ValueError:
            print(f"集合 {collection_name} 不存在")
            return 0

    def create_collection(self, collection_name: str) -> Collection:
        return self.client.get_or_create_collection(
            name=collection_name,
            embedding_function=self.embedding_function
        )


    def get_max_doc_id(self, collection: Collection) -> int:
        """获取集合中现有文档的最大ID序号（假设ID格式为 doc_数字）"""
        # 获取所有文档ID
        all_ids = collection.get()["ids"]
        if not all_ids:
            return -1  # 若集合为空，从0开始
        
        # 提取ID中的数字（例如从 "doc_3" 中提取 3）
        max_id = -1
        pattern = re.compile(r"doc_(\d+)")  # 匹配 "doc_数字" 格式
        for doc_id in all_ids:
            match = pattern.match(doc_id)
            if match:
                current_id = int(match.group(1))
                if current_id > max_id:
                    max_id = current_id
        return max_id

    def split_text(self, text: str) -> List[str]:
        """使用 tiktoken 分块（按字符数拆分，保留语义）"""
        chunks=text.split('\n\n')
        chunks= [chunk.strip() for chunk in chunks if chunk.strip()]
        return chunks

    # 以下方法与之前相同（省略，保持不变）
    def add_texts_to_collection(self, collection: Collection, texts: List[str], metadatas: List[Dict[str, Any]] = None) -> None:
        # 获取当前最大ID序号
        max_id = self.get_max_doc_id(collection)

        ids = [f"doc_{max_id+1+i}" for i in range(len(texts))]
        collection.add(
            documents=texts,
            metadatas=metadatas if metadatas else [{} for _ in range(len(texts))],
            ids=ids
        )
        print(f"成功写入 {len(texts)} 条文本数据到集合 {collection.name}")

    def delete_documents_by_ids(self, collection: Collection, ids: List[str]) -> None:
        """根据ID删除文档（支持单个或多个ID）"""
        if not ids:
            print("请提供要删除的文档ID列表")
            return
        # 执行删除
        collection.delete(ids=ids)
        print(f"成功删除 {len(ids)} 条文档（ID: {ids}）")

    def load_all_documents(self, collection: Collection) -> List[Dict[str, Any]]:
        result = collection.get() # 获取文档
        return [
            {"id": id, "document": doc, "metadata": meta} 
            for id, doc, meta in zip(result["ids"], result["documents"], result["metadatas"])
        ]
    # 根据id获取文档
    def get_document_vectors(self, collection: Collection, ids: List[str]) -> List[Dict[str, Any]]:
        """根据文档ID获取对应的向量及文档内容"""
        if not ids:
            return []
        # 调用get方法，指定需要返回embeddings（向量）和documents（文档内容）
        result = collection.get(ids=ids, include=["embeddings", "documents"])
        # 格式化结果：每个元素包含id、document、embedding
        return [
            {
                "id": doc_id,
                "document": doc,
                "embedding": embedding  # 向量列表（float类型）
            } for doc_id, doc, embedding in zip(
                result["ids"],
                result["documents"],
                result["embeddings"]
            )
        ]

    def similarity_search(self, collection: Collection, query: str, top_k: int = TOP_K) -> List[Dict[str, Any]]:
        results = collection.query(
            query_texts=[query],
            n_results=top_k,
            include=["documents", "metadatas", "distances"]
        )
        return [
            {
                "document": doc,
                "metadata": meta,
                "distance": dist,
                "similarity": 1 - dist if dist <= 1 else 0
            } for doc, meta, dist in zip(
                results["documents"][0],
                results["metadatas"][0],
                results["distances"][0]
            )
        ]


In [4]:
# 删除collection
client = PersistentClient(path=CHROMA_DB_PATH)
client.delete_collection("weiquan")

In [ ]:
# 导入问答对
kb_manager = ChromaKBManager()
collection = kb_manager.create_collection("weiquan")

# sample_texts = [
#     """人工智能（AI）是计算机科学的一个分支，致力于创造能够模拟人类智能的系统。
#     其核心领域包括机器学习、自然语言处理、计算机视觉等。近年来，大语言模型的发展
#     推动了AI在对话系统、内容生成等领域的应用。""",
#     """机器学习是人工智能的子集，专注于开发能从数据中学习的算法。监督学习、无监督学习
#     和强化学习是其主要范式。常见算法包括决策树、支持向量机、神经网络等。深度学习是
#     机器学习的一个分支，基于多层神经网络实现复杂特征提取。""",
#     """自然语言处理（NLP）研究计算机与人类语言的交互。关键任务包括文本分类、机器翻译、
#     情感分析、问答系统等。预训练语言模型（如BERT、GPT）通过大规模文本训练，显著提升了
#     NLP任务的性能，使得机器能够更准确地理解和生成人类语言。"""
# ]


with open(r'/home/dell/Documents/问答对-所有4.txt', 'r', encoding='utf-8') as file:
    text = file.read()

all_chunks = []
all_metadatas = []
for idx, text in enumerate([text]): # 这里的编号要与之前的数据衔接
    chunks = kb_manager.split_text(text) # 使用知识库的方法实现文本分块
    all_chunks.extend(chunks)
    all_metadatas.extend([{"source": f"text_{idx}"} for _ in range(len(chunks))])

kb_manager.add_texts_to_collection(collection, all_chunks, all_metadatas)

all_docs = kb_manager.load_all_documents(collection)
print(f"\n集合中共有 {len(all_docs)} 个文本块")

# 集合中共有 2023 个文本块
# 下一步 分词

# for doc in all_docs:
#     print(f"ID: {doc['id']}, 内容: {doc['document'][:100]}...")

# queries = [
#     "什么是机器学习？",
#     "自然语言处理有哪些应用？",
#     "大语言模型属于人工智能的哪个领域？"
# ]

# for query in queries:
#     print(f"\n查询: {query}")
#     print("相似结果:")
#     results = kb_manager.similarity_search(collection, query)
#     for i, res in enumerate(results, 1):
#         print(f"\n第{i}名 (相似度: {res['similarity']:.2f}):")
#         print(f"内容: {res['document']}")
#         print(f"来源: {res['metadata']}")

/home/dell/anaconda3/envs/vllm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/dell/anaconda3/envs/vllm/lib/python3.11/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


成功写入 4021 条文本数据到集合 weiquan

集合中共有 4021 个文本块


In [ ]:
# 文档分词  关键词
import jieba
with open("/home/dell/Documents/stop_words-utf-8.txt", "r", encoding="utf-8") as f:
    stopwords = set(line.strip() for line in f if line.strip())

punctuations=set('。？！，、；：“”‘’（）〔〕【】《》〈〉——……—～·．＿〝〞｛｝〔〕「」『』・｜…—＿')

def process_text(text: str) -> str:
    # 分词
    words = jieba.lcut(text)
    # 过滤停用词和标点符号
    filtered_words = [
        word for word in words
        if word not in stopwords
        and word not in punctuations
        and len(word) > 1  # 过滤单字
    ]
    return ' '.join(filtered_words)


# 更新数据
kb_manager = ChromaKBManager()
collection = kb_manager.create_collection("weiquan")
all_docs = kb_manager.load_all_documents(collection)

doc_ids=[]
metadata=[]
for doc in all_docs:
    doc_ids.append(doc['id'])
    metadata.append({'words':process_text(doc['document'])})    

collection.update(
    ids=doc_ids,
    metadatas=metadata
)

Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 0.639 seconds.
Prefix dict has been built successfully.


In [ ]:
# 确认结果
kb_manager = ChromaKBManager()
collection = kb_manager.create_collection("weiquan")
# collection.count()
all_docs = kb_manager.load_all_documents(collection)
all_docs[1000]  # 获取第一个文档的内容



{'id': 'doc_0',
 'document': '当前网购中出现的‘信心诈骗’有哪些典型特征？\n信心诈骗的典型特征包括：骗子在淘宝、抖音、快手等平台开设网店，故意发错货或卖次品；当消费者联系客服时，谎称提供售后补偿，诱导消费者扫描快递上的二维码或添加微信；一旦脱离官方平台，便以‘做任务赚钱’（如小额充值看短剧）为诱饵，先返少量佣金获取信任，随后诱使消费者加大投入，并以操作失误、账号溶解等借口要求转账解冻，最终骗光钱财。[1]',
 'metadata': {'source': 'text_0',
  'words': '当前 网购 出现 信心 诈骗 典型 特征 信心 诈骗 典型 特征 包括 骗子 淘宝 抖音 快手 平台 开设 网店 发错 次品 消费者 联系 客服 谎称 提供 售后 补偿 诱导 消费者 扫描 快递 二维码 添加 微信 脱离 官方 平台 任务 赚钱 小额 充值 短剧 诱饵 先返 少量 佣金 获取 信任 随后 诱使 消费者 加大 投入 操作失误 账号 溶解 借口 要求 转账 解冻 最终 骗光 钱财'}}

In [9]:
all_docs[1000]  # 获取第一个文档的内容

{'id': 'doc_1000',
 'document': '商家因报复消费者可能面临哪些法律责任？\n可能被公安机关处以拘留、罚款等行政处罚；若提起民事诉讼，还可能被判令停止侵权并赔偿损失。[119]',
 'metadata': {'words': '商家 报复 消费者 面临 法律责任 公安机关 处以 拘留 罚款 行政处罚 提起 民事诉讼 判令 停止 侵权 赔偿损失 119',
  'source': 'text_0'}}

In [18]:
kb_manager = ChromaKBManager()
collection = kb_manager.create_collection("weiquan")

# 提取所有问题
all_docs = kb_manager.load_all_documents(collection)

# list1='''消费者收到声称可通过12315平台退费的快递或信息，应如何判断真伪？
# 当前出现的假冒12315维权诈骗有哪些典型特征？
# 正规的退费流程是否会要求消费者先购买理财产品或支付费用？
# 消费者在维权过程中应如何保护个人隐私信息？
# 如果不慎被假冒12315退费诈骗骗钱，应采取哪些紧急措施？
# 为什么骗子会利用12315平台进行诈骗？
# 消费者应通过什么渠道下载12315官方APP？'''


list1='''普通消费者在遭遇黑恶势力侵害时，应拨打哪个电话进行举报？
如果消费者报警后问题未得到有效解决，还有哪些官方渠道可以进一步反映问题？
消费者在维权过程中涉及法院相关事务（如案件咨询、诉讼流程等），应拨打哪个电话？
文中提到的三个电话号码分别对应哪些维权场景？
无权无势的普通人在维权时是否缺乏有效渠道？'''

list1 = list1.split('\n')
list1

ids=[]

for doc in all_docs:
    # print(repr(doc['document']))
    doctxt=doc['document'].split('\n')[0]
    if doctxt in list1:
        ids.append(doc['id'])
        print(doc['id'],doctxt)  # 打印前100个字符



doc_1429 普通消费者在遭遇黑恶势力侵害时，应拨打哪个电话进行举报？
doc_1430 如果消费者报警后问题未得到有效解决，还有哪些官方渠道可以进一步反映问题？
doc_1431 消费者在维权过程中涉及法院相关事务（如案件咨询、诉讼流程等），应拨打哪个电话？
doc_1432 文中提到的三个电话号码分别对应哪些维权场景？
doc_1433 无权无势的普通人在维权时是否缺乏有效渠道？
doc_1434 普通消费者在遭遇黑恶势力侵害时，应拨打哪个电话进行举报？
doc_1435 如果消费者报警后问题未得到有效解决，还有哪些官方渠道可以进一步反映问题？
doc_1436 消费者在维权过程中涉及法院相关事务（如案件咨询、诉讼流程等），应拨打哪个电话？
doc_1437 文中提到的三个电话号码分别对应哪些维权场景？
doc_1438 无权无势的普通人在维权时是否缺乏有效渠道？


In [19]:
# ids='''doc_1031
# doc_1032
# doc_1033
# doc_1034
# doc_1035
# doc_1036
# doc_1037'''

ids='''doc_1434
doc_1435
doc_1436
doc_1437
doc_1438
'''

ids = ids.split('\n')
ids


# 执行删除
collection.delete(ids=ids)
print(f"成功删除 {len(ids)} 条文档（ID: {ids}）")

成功删除 6 条文档（ID: ['doc_1434', 'doc_1435', 'doc_1436', 'doc_1437', 'doc_1438', '']）


In [12]:
# repr()
print(repr('ddd\n'))

'ddd\n'


In [10]:
# 提取所有问题
all_docs = kb_manager.load_all_documents(collection)
with open('questions.txt', 'w') as f:
    for doc in all_docs:
        f.write(doc['document'].split('\n')[0]+'\n')
        # print(doc['document'].split('\n')[0]) 

In [ ]:
# 删除数据
def delete_documents_by_ids(self, collection: Collection, ids: List[str]) -> None:
    """根据ID删除文档（支持单个或多个ID）"""
    if not ids:
        print("请提供要删除的文档ID列表")
        return
    # 执行删除
    collection.delete(ids=ids)
    print(f"成功删除 {len(ids)} 条文档（ID: {ids}）")

In [ ]:
# 单个文档的内容
kb_manager = ChromaKBManager()
collection = kb_manager.create_collection("weiquan")
all_docs = kb_manager.load_all_documents(collection)
all_docs[0]

/home/dell/anaconda3/envs/vllm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'id': 'doc_0',
 'document': '当前网购中出现的‘信心诈骗’有哪些典型特征？\n信心诈骗的典型特征包括：骗子在淘宝、抖音、快手等平台开设网店，故意发错货或卖次品；当消费者联系客服时，谎称提供售后补偿，诱导消费者扫描快递上的二维码或添加微信；一旦脱离官方平台，便以‘做任务赚钱’（如小额充值看短剧）为诱饵，先返少量佣金获取信任，随后诱使消费者加大投入，并以操作失误、账号溶解等借口要求转账解冻，最终骗光钱财。',
 'metadata': {'words': '当前 网购 出现 信心 诈骗 典型 特征 信心 诈骗 典型 特征 包括 骗子 淘宝 抖音 快手 平台 开设 网店 发错 次品 消费者 联系 客服 谎称 提供 售后 补偿 诱导 消费者 扫描 快递 二维码 添加 微信 脱离 官方 平台 任务 赚钱 小额 充值 短剧 诱饵 先返 少量 佣金 获取 信任 随后 诱使 消费者 加大 投入 操作失误 账号 溶解 借口 要求 转账 解冻 最终 骗光 钱财',
  'source': 'text_0'}}

In [ ]:
# 相似文档查询
from rank_bm25 import BM25Okapi

corpus=[]
doc_ids=[]
doc_content=[]
source=[]
for doc in all_docs:
    doc_ids.append(doc['id'])
    source.append(doc['source'])
    doc_content.append(doc['document'])
    corpus.append(doc['metadata']['words'])    

bm25 = BM25Okapi(corpus)
query='假一罚十'
doc_scores = bm25.get_scores(query)
print(doc_scores)

# 查找最相似的文档
doc_with_scores = list(zip(range(len(corpus)), doc_scores))
# 按分数降序排序
sorted_docs = sorted(doc_with_scores, key=lambda x: x[1], reverse=True)

# 8. 输出结果
print(f"查询关键词：{query}\n")
print("匹配结果（按相关性排序）：")
for idx, score in sorted_docs[:10]:
    print(f"文档{idx+1}（分数：{score:.4f}）：{doc_content[idx]}")

查询关键词：假一罚十

匹配结果（按相关性排序）：
文档814（分数：10.0666）：如果商家承诺的赔偿标准（如‘假一赔十’）高于法定标准，是否必须兑现？
是的。商家承诺的赔偿标准高于法定赔偿标准的，必须按照其承诺进行赔偿，例如‘假一赔十’可能真的要假一赔十。
文档287（分数：9.3400）：拼多多客服承认“假一赔十”是平台与商家共同承诺后，消费者能否据此要求平台直接赔付？
可以。既然“假一赔十”是平台与商家“共同推出”的承诺，平台就负有履约义务。当商品被证实存在冒用品牌、虚假宣传等售假行为时，消费者有权要求拼多多平台直接承担责任，兑现“假一赔十”的赔偿，而不是仅让商家处理或推卸责任。
文档283（分数：9.3351）：拼多多平台上的“假一赔十”承诺是如何规定的？由谁承担责任？
根据短文，拼多多客服表示“假一赔十”（“假衣陪食”）是“商家跟我们平台共同推出的”承诺。这意味着该承诺由平台与商家共同作出，因此平台有义务在商家售假时对消费者负责，不能将责任完全推给商家。一旦核实存在“未经品牌许可、冒用他人品牌”的假货情况，平台应支持消费者售后并兑现“假一赔十”的赔偿。
文档479（分数：9.0651）：经营者销售明知是假冒伪劣保健食品应承担什么责任？
应承担惩罚性赔偿责任，包括退还货款并支付价款十倍的赔偿金，以打击和遏制销售假冒伪劣保健食品的违法行为。
文档182（分数：9.0353）：所有假货都能主张“假一赔十”吗？
不能。‘假一赔十’仅适用于《食品安全法》规定的不符合食品安全标准的食品，赔偿为价款十倍或损失三倍，不足1000元按1000元计；其他商品若存在欺诈行为，适用《消费者权益保护法》，赔偿为价款三倍，不足500元按500元计。
文档185（分数：8.9876）：如果商家承诺‘假一赔十’但商品不属于食品，该承诺是否有效？
有效。按照‘约定优于法定’原则，即使法律未规定该类商品适用‘假一赔十’，只要商家事先作出此承诺，就应当兑现。
文档183（分数：8.8572）：食品类假货的惩罚性赔偿标准是什么？
根据《食品安全法》，消费者可要求支付价款十倍或损失三倍的赔偿金，增加赔偿金额不足1000元的，按1000元赔偿。
文档1347（分数：8.7559）：商家以“拆封即视为影响二次销售”为由拒绝退货，可能面临什么法律后果？
根据《侵害消费者权益行为处罚办法》第九条，商家

In [ ]:
all_docs = kb_manager.load_all_documents(collection) 
print(f"\n集合中共有 {len(all_docs)} 个文本块")
for doc in all_docs:
    print(f"ID: {doc['id']}, 内容: {doc['document'][:100]}...")


集合中共有 1439 个文本块
ID: doc_0, 内容: 当前网购中出现的‘信心诈骗’有哪些典型特征？
信心诈骗的典型特征包括：骗子在淘宝、抖音、快手等平台开设网店，故意发错货或卖次品；当消费者联系客服时，谎称提供售后补偿，诱导消费者扫描快递上的二维码或添加...
ID: doc_1, 内容: 消费者在网购后遇到商家引导脱离平台沟通售后，应如何应对？
消费者在网购后一定不要脱离官方平台与商家沟通，应始终在淘宝、抖音、快手等平台的官方客服渠道内处理售后问题，避免扫描快递上的二维码或添加商家微信...
ID: doc_2, 内容: 为什么不能扫描快递上的二维码或添加商家微信处理售后？
因为这是骗子常用的诈骗手段。一旦通过扫描快递二维码或添加微信脱离官方平台，消费者将失去平台保护，骗子会以做任务、充值返佣等方式诱导转账，最终以操作...
ID: doc_3, 内容: 骗子在电商平台实施诈骗的具体步骤是什么？
骗子在淘宝、抖音、快手等平台开网店，故意发错货或卖次品；等消费者找客服时，谎称提供售后补偿，引导其扫描快递二维码或加微信；脱离平台后，让受害者做任务（如小额充...
ID: doc_4, 内容: 消费者在网购维权过程中应坚持什么基本原则？
消费者在网购维权过程中应始终坚持在官方平台内沟通和处理售后问题，不要脱离淘宝、抖音、快手等平台的官方客服渠道，避免通过快递二维码、私人微信等外部方式联系商家...
ID: doc_5, 内容: 如何识别商家是否在实施‘信心诈骗’？
若商家在发错货或卖次品后主动提出‘售后补偿’，并引导消费者扫描快递上的二维码或添加微信等脱离平台的行为，就极有可能是信心诈骗。此外，若对方提出‘看短剧赚钱’‘做任...
ID: doc_6, 内容: 消费者在遭遇发错货或收到次品时，正确的维权方式是什么？
消费者应直接通过淘宝、抖音、快手等平台的官方客服渠道反馈问题并申请售后，切勿接受商家引导去扫描快递二维码或添加微信等脱离平台的沟通方式，以确保自...
ID: doc_7, 内容: 在消费维权过程中，如何提高维权成功率？
在投诉的同时一定要同时举报商家的不法行为，要求责令改正并予以处罚。...
ID: doc_8, 内容: 消费者在维权时除了投诉还应该做什么？
在投诉的同时一定要同时举报商家的不法行为，要求责令改正并予以处罚。...
ID: doc

In [73]:
kb_manager = ChromaKBManager()
collection = kb_manager.create_collection("weiquan")

queries = [
    "什么情况适用假一罚十？"
]

for query in queries:
    # print(f"\n查询: {query}")
    # print("相似结果:")
    results = kb_manager.similarity_search(collection, query,top_k=5)
    # for i, res in enumerate(results, 1):
    #     print(f"\n第{i}名 (相似度: {res['similarity']:.2f}):")
    #     print(f"内容: {res['document']}")
    #     print(f"来源: {res['metadata']}")
    retrieve_info=''
    for i, res in enumerate(results, 1):
        retrieve_info+=f"检索结果{i}： \n{res['document']}\n\n"
    # print(retrieve_info)

    prompt=f'''
    ## 用户问题是：{query}

    ## 针对此问题，在知识库中找到如下信息：
    """
    {retrieve_info}
    """
    ## 回答要求：
    - 生成回复时，只能基于上述提供的检索结果，绝对不能使用外部知识
    - 对上述检索结果进行系统化整理，使生成回复更加有条理
    '''

    # print(prompt)
    call_llm(prompt)



<think>
好的，我现在需要处理用户的问题：“什么情况适用假一罚十？”根据提供的检索结果，我需要整理出适用的情况，并确保不使用外部知识。

首先，查看检索结果1，里面提到“假一赔十”仅适用于《食品安全法》中的不符合安全标准的食品，赔偿是价款十倍或损失三倍，不足1000元按1000元算。其他商品如果有欺诈行为，适用《消费者权益保护法》，是价款三倍，不足500元按500元算。这说明食品领域有特别规定，而其他商品可能不同。

接着，检索结果2指出，如果商家承诺“假一赔十”但商品不是食品，这个承诺仍然有效，因为约定优于法定。所以即使法律没有规定，商家的承诺也必须兑现。这可能扩大了适用范围，但仅限于商家主动承诺的情况。

检索结果3区分了“退一赔三”和“十倍赔偿”的区别，前者适用于欺诈行为，依据《消费者权益保护法》，后者适用于食品不符合安全标准，依据《食品安全法》。这里再次确认了食品领域的十倍赔偿，而其他欺诈情况是三倍。

检索结果4提到消费者如果误以为宠物食品适用一赔十，应转而使用《消费者权益保护法》来主张三倍赔偿，前提是商家有欺诈行为。这说明宠物食品可能属于非食品类，适用不同的法律。

检索结果5涉及拼多多平台的“假一赔十”承诺，由平台和商家共同负责。这说明在特定平台购买时，如果商家售假，平台也要承担赔偿责任，这可能是一个特殊情况。

现在需要将这些信息系统化整理。首先，分情况讨论：食品和其他商品。对于食品，适用《食品安全法》的十倍赔偿；其他商品如果有欺诈行为，适用《消费者权益保护法》的三倍赔偿。但如果有商家承诺“假一赔十”，即使不是食品，承诺也有效。另外，拼多多平台的情况由平台和商家共同承担责任。

需要注意的是，不能混淆不同法律条款的应用范围，比如宠物食品可能属于其他商品，适用三倍赔偿而非十倍。同时，商家的承诺可以覆盖法律未规定的情况，但仅限于其承诺的范围。

总结时，应分点列出适用情况，包括食品、其他商品的欺诈、商家承诺、平台责任等，并引用对应的法律条款和检索结果中的信息，确保准确无误。
</think>

根据提供的检索结果，适用“假一罚十”的情形可系统化整理如下：

---

### **一、适用“假一赔十”的法定情形**
1. **食品领域（《食品安全法》适用）**  
   - **适用条件**：商品为**不符合食品安全标准的食品**（如掺假、过期、有害

In [58]:
info_retrieved='''第4名 (相似度: 0.58):
内容: 消费者误以为宠物食品适用‘一赔十’规定，应如何正确维权？
应放弃依据《食品安全法》主张一赔十，转而依据《消费者权益保护法》，在商家存在欺诈或明知销售过期商品的情形下，主张三倍价款或两倍损失以下的惩罚性赔偿，并要求退货退款。
来源: {'source': 'text_0'}

第5名 (相似度: 0.57):
内容: 拼多多平台上的“假一赔十”承诺是如何规定的？由谁承担责任？
根据短文，拼多多客服表示“假一赔十”（“假衣陪食”）是“商家跟我们平台共同推出的”承诺。这意味着该承诺由平台与商家共同作出，因此平台有义务在商家售假时对消费者负责，不能将责任完全推给商家。一旦核实存在“未经品牌许可、冒用他人品牌”的假货情况，平台应支持消费者售后并兑现“假一赔十”的赔偿。
来源: {'source': 'text_0'}'''
query='图书丢失如何赔偿'

prompt=f'''
## 用户问题是：{query}

## 知识库中找到如下信息：
"
{info_retrieved}
"
## 回答要求：
- 生成回复时，仅使用找到的知识库信息
- 对找到的知识库信息进行系统化整理，使内容更加有条理
'''

print(prompt)


## 用户问题是：图书丢失如何赔偿

## 知识库中找到如下信息：
"
第4名 (相似度: 0.58):
内容: 消费者误以为宠物食品适用‘一赔十’规定，应如何正确维权？
应放弃依据《食品安全法》主张一赔十，转而依据《消费者权益保护法》，在商家存在欺诈或明知销售过期商品的情形下，主张三倍价款或两倍损失以下的惩罚性赔偿，并要求退货退款。
来源: {'source': 'text_0'}

第5名 (相似度: 0.57):
内容: 拼多多平台上的“假一赔十”承诺是如何规定的？由谁承担责任？
根据短文，拼多多客服表示“假一赔十”（“假衣陪食”）是“商家跟我们平台共同推出的”承诺。这意味着该承诺由平台与商家共同作出，因此平台有义务在商家售假时对消费者负责，不能将责任完全推给商家。一旦核实存在“未经品牌许可、冒用他人品牌”的假货情况，平台应支持消费者售后并兑现“假一赔十”的赔偿。
来源: {'source': 'text_0'}
"
## 回答要求：
- 生成回复时，仅使用找到的知识库信息
- 对找到的知识库信息进行系统化整理，使内容更加有条理



In [16]:
client1 = PersistentClient(path=CHROMA_DB_PATH)
collections = client1.list_collections()
a=[col.name for col in collections]
print(a)

NameError: name 'PersistentClient' is not defined

In [15]:
collection = client1.get_collection(name='example_kb2')
collection.count()  # 返回文档总数

NameError: name 'client1' is not defined

In [18]:
all_docs = kb_manager.load_all_documents(collection)
print(f"\n集合中共有 {len(all_docs)} 个文本块")
for doc in all_docs:
    print(f"ID: {doc['id']}, 内容: {doc['document'][:100]}...")



集合中共有 3 个文本块
ID: doc_0, 内容: 人工智能（AI）是计算机科学的一个分支，致力于创造能够模拟人类智能的系统。
    其核心领域包括机器学习、自然语言处理、计算机视觉等。近年来，大语言模型的发展
    推动了AI在对话系统、内容生成...
ID: doc_1, 内容: 机器学习是人工智能的子集，专注于开发能从数据中学习的算法。监督学习、无监督学习
    和强化学习是其主要范式。常见算法包括决策树、支持向量机、神经网络等。深度学习是
    机器学习的一个分支，基于...
ID: doc_2, 内容: 自然语言处理（NLP）研究计算机与人类语言的交互。关键任务包括文本分类、机器翻译、
    情感分析、问答系统等。预训练语言模型（如BERT、GPT）通过大规模文本训练，显著提升了
    NLP任务...


In [16]:
result = collection.get(ids='doc_0', include=["embeddings", "documents"])
result

{'ids': ['doc_0'],
 'embeddings': array([[ 0.00858453,  0.00275543, -0.01579159, ..., -0.00704395,
          0.0377815 , -0.02407671]], shape=(1, 1024)),
 'documents': ['人工智能（AI）是计算机科学的一个分支，致力于创造能够模拟人类智能的系统。\n    其核心领域包括机器学习、自然语言处理、计算机视觉等。近年来，大语言模型的发展\n    推动了AI在对话系统、内容生成等领域的应用。'],
 'uris': None,
 'included': ['embeddings', 'documents'],
 'data': None,
 'metadatas': None}

In [44]:
CHROMA_DB_PATH = "./chroma_local_db"
client1 = PersistentClient(path=CHROMA_DB_PATH)
collections = client1.list_collections()

# for col in collections:
#     client1.delete_collection(name=col.name)

a=[col.name for col in collections]
print(a)


[]


In [ ]:
# 删除文档集合
client1.delete_collection(name="weiquan")

## call ollama api: not stream

## call ollama api: stream